# EDA — Netflix Movies and TV Shows

**Dataset:** Netflix Movies and TV Shows (Kaggle)  
**Aluno:** Matheos Guerra  
**Disciplina:** Análise e Visualização de Dados  

Este notebook aplica o ciclo completo de EDA ao dataset *Netflix Movies and TV Shows*, replicando a estrutura demonstrada em aula com o dataset Heart Disease (UCI). O dataset contém informações sobre títulos disponíveis na plataforma Netflix, incluindo filmes e séries.

**Variáveis principais:**
- `show_id` — identificador único
- `type` — Movie ou TV Show (**variável de grupo principal**)
- `title` — título
- `director`, `cast` — diretor(es) e elenco
- `country` — país(es) de origem
- `date_added` — data de adição na plataforma
- `release_year` — ano de lançamento
- `rating` — classificação etária
- `duration` — duração (min para filmes; temporadas para séries)
- `listed_in` — gênero(s)
- `description` — sinopse

**Convenção de cores (análoga à aula):**
- Rosa-claro → *Movie* (classe de referência)
- Roxo → *TV Show* (classe de interesse)
- Cinza → visualizações sem separação por grupo
- Vermelho → outliers / pontos de atenção

## Setup de Bibliotecas e Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import entropy

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

COR_NEUTRA   = "lightgray"
COR_CONTEXTO = "#e9d4d0"   # rosa-claro → Movie
COR_DESTAQUE = "#2d223a"   # roxo       → TV Show
COR_ALERTA   = "tab:red"

palette_tipo = {"Movie": COR_CONTEXTO, "TV Show": COR_DESTAQUE}

## Carregando o Dataset

In [ ]:
df = pd.read_csv("../../datasetnetflix/netflix_titles.csv")
df.head()

## 1. Princípio Estrutural

Antes de qualquer análise, precisamos entender a "anatomia" dos dados: quantas linhas e colunas temos, que tipo de dado cada coluna guarda, se há valores faltantes e se há duplicadas.

**Antes de rodar:** o dataset Netflix reúne o catálogo completo de títulos disponíveis na plataforma. Quantos títulos você esperaria encontrar — centenas, alguns milhares, dezenas de milhares?

### Dimensão e Tipos de Dados

In [ ]:
print("Dimensão:", df.shape)
print(f"\n{df.shape[0]} linhas × {df.shape[1]} colunas")

In [ ]:
df.info()

**Discussão:** Quais colunas o pandas inferiu como `object` (texto)? O campo `duration` aparece como texto — por que isso é um problema para análises numéricas?

In [ ]:
df.describe()

**Discussão:** O `describe()` padrão do pandas só mostra estatísticas para colunas numéricas. Aqui temos apenas `release_year`. O que isso nos diz sobre o perfil deste dataset?

### Duplicadas

In [ ]:
n_dup = df.duplicated().sum()
print(f"Linhas duplicadas: {n_dup}")

# Verificando show_id (identificador único esperado)
n_dup_id = df["show_id"].duplicated().sum()
print(f"show_id duplicados: {n_dup_id}")

### Distribuição entre as duas classes principais

In [ ]:
contagem_tipo = df["type"].value_counts()
print(contagem_tipo)
print(f"\nProporção:\n{(contagem_tipo / len(df) * 100).round(1)}")

## 2. Dados Faltantes

### Quantificando os ausentes

In [ ]:
ausentes = df.isna().sum()
ausentes_pct = (ausentes / len(df) * 100).round(2)

resumo_ausentes = pd.DataFrame({
    "ausentes": ausentes,
    "percentual": ausentes_pct
}).sort_values("percentual", ascending=False)

resumo_ausentes[resumo_ausentes["ausentes"] > 0]

**Discussão:** Quais colunas têm mais ausentes? A ausência de `director` (~30%) faz sentido intuitivo para algum tipo de conteúdo? E a ausência de `rating`?

#### Identificando o mecanismo de dados ausentes

**Conhecimento de Domínio**

- `director` (~30% ausente): Séries de TV frequentemente possuem múltiplos diretores por episódio, tornando o preenchimento desse campo impraticável. Também é comum em reality shows, documentários de entrevistas e produções internacionais. → Mecanismo provável: **MAR** (Missing At Random) — a ausência está relacionada ao *tipo* de conteúdo, não ao diretor em si.

- `cast` (~9% ausente): Produções animadas, documentários sem entrevistas ou conteúdo infantil educativo podem não ter elenco listado. → **MAR**.

- `country` (~6% ausente): Produções de origem incerta ou co-produções não atribuídas. → Provável **MAR**.

- `date_added` (~5% ausente): Títulos que podem ter sido adicionados antes do sistema de registro ser implementado. → Provável **MCAR** (Missing Completely At Random).

- `rating` (~3% ausente): Classificação etária não definida — pode ocorrer para conteúdo em processo de catalogação. → Provável **MAR**.

*Relembrando:*
- **MCAR**: ausência completamente aleatória, sem relação com outros dados
- **MAR**: ausência relacionada a outras variáveis observáveis (mas não ao próprio valor ausente)
- **MNAR**: ausência relacionada ao próprio valor que está faltando

#### As ausências ocorrem juntas?

In [ ]:
cols_com_ausentes = df.columns[df.isna().any()].tolist()
print("Colunas com ausentes:", cols_com_ausentes)

# Verificar se ausência de 'director' e 'cast' ocorrem juntas
ambos_ausentes = df["director"].isna() & df["cast"].isna()
print(f"\nLinhas com 'director' E 'cast' ausentes: {ambos_ausentes.sum()}")
print(f"Linhas com 'director' ausente: {df['director'].isna().sum()}")
print(f"Linhas com 'cast' ausente: {df['cast'].isna().sum()}")

In [ ]:
# Relação entre ausência de director e o tipo (Movie vs TV Show)
ausente_dir_por_tipo = df.groupby("type")["director"].apply(lambda x: x.isna().mean() * 100).round(2)
print("% de 'director' ausente por tipo:")
print(ausente_dir_por_tipo)

fig, ax = plt.subplots(figsize=(6, 4))
ausente_dir_por_tipo.plot(kind="bar", color=[COR_CONTEXTO, COR_DESTAQUE], ax=ax)
ax.set_title("% de 'director' ausente por tipo de conteúdo")
ax.set_xlabel("")
ax.set_ylabel("%")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

**Discussão:** A ausência de `director` é significativamente maior para *TV Show* ou *Movie*? Se a ausência se concentrar em um tipo específico, isso confirma a hipótese de mecanismo **MAR** — a ausência não é aleatória, está relacionada a outra variável observável (`type`).

**Nota:** Com um volume grande de ausentes (30% em `director`), a correlação entre indicadores de ausência é mais confiável do que seria com poucos casos. Aqui a análise por tipo (acima) é mais informativa do que uma simples contagem bruta.

## 3. Princípio Distribucional

Ainda com os dados originais (sem tratar ausentes ou outliers), calculamos as estatísticas descritivas e olhamos as distribuições. Esse retrato "antes" orientará as decisões de tratamento.

### Preparação: Extraindo Variáveis Numéricas

O dataset tem poucas variáveis genuinamente numéricas. Precisamos extrair algumas:

In [ ]:
# Extrair número de 'duration' (ex: '90 min' → 90; '2 Seasons' → 2)
df["duration_num"] = df["duration"].str.extract(r"(\d+)").astype(float)

# Extrair unidade de duração
df["duration_unit"] = df["duration"].str.extract(r"(\D+)").str.strip()

# Extrair ano e mês de adição
df["date_added_parsed"] = pd.to_datetime(df["date_added"].str.strip(), format="%B %d, %Y", errors="coerce")
df["year_added"]  = df["date_added_parsed"].dt.year
df["month_added"] = df["date_added_parsed"].dt.month

# Confirmar extração
print("Tipos de duration_unit:\n", df["duration_unit"].value_counts())
print("\nExemplos:")
df[["title", "type", "duration", "duration_num", "duration_unit", "year_added"]].sample(5, random_state=42)

**Atenção:** `duration_num` mistura duas métricas incomparáveis — *minutos* para filmes e *temporadas* para séries. Para análise distribucional correta, precisamos separar por tipo.

In [ ]:
df_movies = df[df["type"] == "Movie"].copy()
df_tv     = df[df["type"] == "TV Show"].copy()

# Variáveis quantitativas
VARS_QUANT_GLOBAL = ["release_year", "year_added"]

print(f"Filmes: {len(df_movies)} | Séries: {len(df_tv)}")

### Tabela Descritiva com Assimetria e Curtose

In [ ]:
desc = df[VARS_QUANT_GLOBAL].describe().T
desc["skewness"] = df[VARS_QUANT_GLOBAL].skew()
desc["kurtosis"] = df[VARS_QUANT_GLOBAL].kurtosis()

(
    desc[["mean", "50%", "std", "min", "max", "skewness", "kurtosis"]]
    .rename(columns={"50%": "median"})
    .round(2)
    .style.format(precision=2)
    .background_gradient(cmap="magma", subset=["skewness", "kurtosis"])
)

In [ ]:
# Duração dos filmes separadamente
desc_dur = df_movies[["duration_num"]].describe().T
desc_dur["skewness"] = df_movies[["duration_num"]].skew()
desc_dur["kurtosis"] = df_movies[["duration_num"]].kurtosis()

(
    desc_dur[["mean", "50%", "std", "min", "max", "skewness", "kurtosis"]]
    .rename(columns={"50%": "median"})
    .round(2)
    .style.format(precision=2)
    .background_gradient(cmap="magma", subset=["skewness", "kurtosis"])
)

**Discussão:** Olhando a tabela, `release_year` apresenta assimetria negativa (cauda para a esquerda) ou positiva? Isso faz sentido intuitivo — a Netflix tem mais títulos antigos ou recentes? E a duração dos filmes, como se comporta?

### Distribuições das Variáveis Quantitativas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(data=df, x="release_year", kde=True, color=COR_NEUTRA, ax=axes[0])
axes[0].set_title("release_year (todos os títulos)")

sns.histplot(data=df.dropna(subset=["year_added"]), x="year_added", kde=True, color=COR_NEUTRA, ax=axes[1])
axes[1].set_title("year_added (quando entrou na Netflix)")

sns.histplot(data=df_movies.dropna(subset=["duration_num"]), x="duration_num", kde=True, color=COR_NEUTRA, ax=axes[2])
axes[2].set_title("duration_num (minutos — somente filmes)")

fig.suptitle("Distribuição das Variáveis Quantitativas: Netflix (sem separar por tipo)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Distribuições por tipo (Movie vs TV Show)
fig, axes = plt.subplots(len(VARS_QUANT_GLOBAL), 2, figsize=(12, 4 * len(VARS_QUANT_GLOBAL)))

for i, col in enumerate(VARS_QUANT_GLOBAL):
    lim = (df[col].min(), df[col].max())
    for j, (tipo, cor) in enumerate([("Movie", COR_CONTEXTO), ("TV Show", COR_DESTAQUE)]):
        subset = df[df["type"] == tipo]
        sns.histplot(data=subset, x=col, kde=True, color=cor, ax=axes[i, j])
        axes[i, j].set_xlim(lim)
        axes[i, j].set_title(f"{col} — {tipo}")

fig.suptitle("Distribuição por Tipo de Conteúdo (painéis separados, mesma escala)", fontsize=14)
plt.tight_layout()
plt.show()

**Discussão:** Comparando Movie vs. TV Show nos painéis de `release_year`: filmes tendem a ser mais antigos ou mais recentes do que séries? E no `year_added`: qual tipo foi adicionado mais cedo à plataforma?

#### Testes de Aderência à Normalidade

O **Shapiro-Wilk** aceita no máximo 5.000 amostras. Com 8.809 registros, fazemos uma amostragem aleatória. Lembrando: com amostras grandes, o teste tende a rejeitar H0 por desvios triviais — usamos o QQ-Plot como apoio visual.

In [ ]:
resultados_shapiro = []
for col in ["release_year", "year_added"]:
    serie = df[col].dropna()
    amostra = serie.sample(min(5000, len(serie)), random_state=42)
    stat, p = stats.shapiro(amostra)
    resultados_shapiro.append({
        "variavel": col,
        "n_amostra": len(amostra),
        "estatistica": round(stat, 4),
        "p_valor": round(p, 6),
        "rejeita_normalidade": p < 0.05
    })

# Duração dos filmes
serie_dur = df_movies["duration_num"].dropna()
amostra_dur = serie_dur.sample(min(5000, len(serie_dur)), random_state=42)
stat_dur, p_dur = stats.shapiro(amostra_dur)
resultados_shapiro.append({
    "variavel": "duration_num (filmes)",
    "n_amostra": len(amostra_dur),
    "estatistica": round(stat_dur, 4),
    "p_valor": round(p_dur, 6),
    "rejeita_normalidade": p_dur < 0.05
})

pd.DataFrame(resultados_shapiro).set_index("variavel")

### QQ-Plot — `release_year`

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (col, dados) in zip(axes, [
    ("release_year", df["release_year"].dropna()),
    ("year_added", df["year_added"].dropna()),
    ("duration_num (filmes)", df_movies["duration_num"].dropna())
]):
    stats.probplot(dados.sample(min(5000, len(dados)), random_state=42), dist="norm", plot=ax)
    ax.get_lines()[0].set_markerfacecolor(COR_NEUTRA)
    ax.get_lines()[0].set_markeredgecolor(COR_NEUTRA)
    ax.get_lines()[1].set_color(COR_ALERTA)
    ax.set_title(f"QQ-Plot: {col} vs. Normal")

plt.tight_layout()
plt.show()

**Discussão:** Os pontos se afastam da reta vermelha nas extremidades? Para `release_year`, o afastamento à esquerda indica que há poucos títulos muito antigos — uma cauda esquerda pesada. Isso é coerente com o histograma?

#### ECDF — Comparando os tipos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, VARS_QUANT_GLOBAL):
    sns.ecdfplot(data=df.dropna(subset=[col]), x=col, hue="type",
                 palette=palette_tipo, ax=ax)
    ax.set_title(f"ECDF — {col} por tipo")

fig.suptitle("ECDF: Comparando Movie vs. TV Show", fontsize=14)
plt.tight_layout()
plt.show()

**Como ler:** onde as duas curvas se distanciam mais, ali está a faixa do eixo X onde os tipos mais diferem. Para `release_year`: em que intervalo de anos Movie e TV Show mais se separam?

### Métricas de Tendência Central

In [ ]:
media_year = df["release_year"].mean()
mediana_year = df["release_year"].median()

plt.figure(figsize=(10, 5))
sns.histplot(data=df, x="release_year", kde=True, color=COR_NEUTRA)
plt.axvline(media_year, color=COR_ALERTA, linestyle="--", linewidth=2, label=f"Média = {media_year:.0f}")
plt.axvline(mediana_year, color="black", linestyle="-", linewidth=2, label=f"Mediana = {mediana_year:.0f}")
plt.title("Ano de Lançamento: Distribuição com Média e Mediana")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Média: {media_year:.1f} | Mediana: {mediana_year:.0f} | Diferença: {abs(media_year - mediana_year):.1f} anos")

**Discussão:** Média e mediana estão próximas? Se a média for menor do que a mediana, isso indica que a cauda esquerda (títulos mais antigos) está puxando a média para baixo. Nesse caso, qual das duas seria melhor para resumir o "ano típico" de um título Netflix?

### Dispersão

In [ ]:
dispersao = df[VARS_QUANT_GLOBAL].agg(["var", "std"]).T
dispersao["IQR"] = df[VARS_QUANT_GLOBAL].quantile(0.75) - df[VARS_QUANT_GLOBAL].quantile(0.25)
dispersao.round(2)

### Boxplot e Violin Plot por Tipo

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, col in enumerate(VARS_QUANT_GLOBAL):
    sns.boxplot(data=df, x="type", y=col, hue="type",
                palette=palette_tipo, legend=False, ax=axes[0, i])
    axes[0, i].set_title(col)
    axes[0, i].set_xlabel("")

    sns.violinplot(data=df, x="type", y=col, hue="type",
                   palette=palette_tipo, legend=False, ax=axes[1, i])
    axes[1, i].set_xlabel("")

axes[0, 0].set_ylabel("Boxplot")
axes[1, 0].set_ylabel("Violin plot")
fig.suptitle("Dispersão por Tipo de Conteúdo — Boxplot vs. Violin Plot", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Duração dos filmes (em minutos)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=df_movies.dropna(subset=["duration_num"]), y="duration_num",
            color=COR_CONTEXTO,
            flierprops=dict(markerfacecolor=COR_ALERTA, markeredgecolor=COR_ALERTA, marker="o"),
            ax=axes[0])
axes[0].set_title("Duração dos Filmes (min) — Boxplot")

sns.violinplot(data=df_movies.dropna(subset=["duration_num"]), y="duration_num",
               color=COR_CONTEXTO, ax=axes[1])
axes[1].set_title("Duração dos Filmes (min) — Violin")

plt.tight_layout()
plt.show()

**Discussão:** O boxplot de `release_year` mostra uma caixa (IQR) nitidamente diferente entre Movie e TV Show? Isso já é um indício — antes mesmo da análise bivariada formal — de que os filmes no Netflix tendem a ter um perfil de lançamento diferente das séries.

Para `duration_num` dos filmes: onde estão os outliers? Filmes com duração extremamente curta (< 30 min) ou muito longa (> 200 min) — são erros de dados ou conteúdos legítimos (curtas-metragens, filmes épicos)?

## 4. Tratamento de Dados Ausentes e Outliers

Agora que temos o retrato estatístico "antes" (Seção 3), podemos decidir como tratar os dados ausentes e os outliers.

### Detecção de Outliers por IQR

In [ ]:
def limites_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

resumo_outliers = {}
for col, dados in [
    ("release_year", df["release_year"]),
    ("year_added", df["year_added"].dropna()),
    ("duration_num (filmes)", df_movies["duration_num"].dropna())
]:
    lim_inf, lim_sup = limites_iqr(dados)
    n_out = ((dados < lim_inf) | (dados > lim_sup)).sum()
    resumo_outliers[col] = {
        "limite_inferior": round(lim_inf, 1),
        "limite_superior": round(lim_sup, 1),
        "n_outliers": int(n_out),
        "pct_outliers": round(n_out / len(dados) * 100, 2)
    }

pd.DataFrame(resumo_outliers).T

### Visualizando os Outliers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (col, dados) in zip(axes, [
    ("release_year", df["release_year"]),
    ("year_added", df["year_added"]),
    ("duration_num\n(filmes, min)", df_movies["duration_num"])
]):
    sns.boxplot(y=dados, color=COR_NEUTRA,
                flierprops=dict(markerfacecolor=COR_ALERTA, markeredgecolor=COR_ALERTA, marker="o"),
                ax=ax)
    ax.set_title(col)

fig.suptitle("Outliers pela Regra do IQR (pontos em vermelho)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Inspecionando os outliers de duration (filmes extremamente curtos ou longos)
lim_inf_dur, lim_sup_dur = limites_iqr(df_movies["duration_num"].dropna())

print(f"Limite IQR para duração de filmes: [{lim_inf_dur:.0f}, {lim_sup_dur:.0f}] minutos")
print("\n--- Filmes muito curtos (possíveis curtas-metragens) ---")
print(df_movies[df_movies["duration_num"] < lim_inf_dur][["title", "duration", "release_year", "listed_in"]].head(10))

print("\n--- Filmes muito longos ---")
print(df_movies[df_movies["duration_num"] > lim_sup_dur][["title", "duration", "release_year", "listed_in"]].sort_values("duration_num", ascending=False).head(10))

**Discussão:** Os outliers de duração são erros de dados ou filmes legítimos? Curtas-metragens (< 30 min) são gêneros válidos; filmes com 3+ horas existem (documentários, épicos). Isso os classifica como outliers **estatísticos legítimos** — a regra do IQR os marca como fora do padrão, mas eles não violam nenhuma regra física ou de negócio.

| Tipo | Cenário | O que fazer? |
|------|---------|----------------|
| Estatístico | Legítimo (curta-metragem, épico) | Manter / Transformar |
| De domínio | Viola regra física (duração negativa) | Investigar! |
| Erro de dados | Digitação incorreta | Corrigir/Remover |

Para `release_year`: títulos pré-1990 são genuinamente raros no catálogo Netflix, mas não são erros — são clássicos deliberadamente incluídos.

### Decidindo o Tratamento dos Dados Ausentes

In [ ]:
df_antes = df.copy()
linhas_antes = df.shape[0]

# 1. 'rating' ausente (~3%): remover — rating é variável de análise relevante
#    e o mecanismo MAR não depende do próprio valor de rating
df = df.dropna(subset=["rating"]).reset_index(drop=True)

# 2. 'director', 'cast': preencher com 'Não informado' para manter os registros
#    A ausência é legítima (MAR) e remover ~30% dos dados seria perda excessiva
df["director"] = df["director"].fillna("Não informado")
df["cast"]     = df["cast"].fillna("Não informado")

# 3. 'country': preencher com 'Unknown' (mesmo raciocínio)
df["country"]  = df["country"].fillna("Unknown")

# 4. 'date_added': remover linhas sem data (são poucas e year_added é usada na análise)
df = df.dropna(subset=["date_added"]).reset_index(drop=True)

linhas_depois = df.shape[0]
print(f"Linhas antes: {linhas_antes} | Linhas depois: {linhas_depois} | Removidas: {linhas_antes - linhas_depois} ({(linhas_antes-linhas_depois)/linhas_antes*100:.1f}%)")
print(f"\nAusentes restantes:\n{df.isna().sum()[df.isna().sum() > 0]}")

### Decidindo o Tratamento dos Outliers

Diferente dos ausentes, aqui **não vamos remover automaticamente**. Os outliers identificados são:

- **Filmes com duração < limiar IQR**: curtas-metragens legítimos — *manter*.
- **Filmes com duração > limiar IQR**: documentários e filmes épicos — *manter*.
- **`release_year` antigo**: clássicos intencionalmente incluídos — *manter*.

Todos se enquadram como **outliers estatísticos legítimos**. Vamos apenas demonstrar como seria uma transformação logarítmica na duração, caso fosse necessária para algum teste paramétrico.

In [ ]:
df_movies_atual = df[df["type"] == "Movie"].copy()
df_movies_atual["duration_log"] = np.log1p(df_movies_atual["duration_num"])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(data=df_movies_atual.dropna(subset=["duration_num"]), x="duration_num",
             kde=True, color=COR_NEUTRA, ax=axes[0])
axes[0].set_title(f"duration_num original (skew={df_movies_atual['duration_num'].skew():.2f})")

sns.histplot(data=df_movies_atual.dropna(subset=["duration_log"]), x="duration_log",
             kde=True, color=COR_NEUTRA, ax=axes[1])
axes[1].set_title(f"log(1 + duration) (skew={df_movies_atual['duration_log'].skew():.2f})")

plt.suptitle("Transformação Logarítmica na Duração dos Filmes (apenas demonstração)", fontsize=13)
plt.tight_layout()
plt.show()

**Discussão:** A transformação logarítmica reduziu a assimetria? Mantivemos `duration_log` só como demonstração — nas próximas seções continuaremos com os valores originais.

## 5. Estatística Descritiva pt 2

Fechando o ciclo iterativo: tratamos os ausentes e decidimos manter os outliers. Agora rodamos a mesma bateria de estatísticas da Seção 3 para comparar com o "antes".

In [ ]:
desc_depois = df[VARS_QUANT_GLOBAL].describe().T
desc_depois["skewness"] = df[VARS_QUANT_GLOBAL].skew()
desc_depois["kurtosis"] = df[VARS_QUANT_GLOBAL].kurtosis()

(
    desc_depois[["mean", "50%", "std", "min", "max", "skewness", "kurtosis"]]
    .rename(columns={"50%": "median"})
    .round(2)
    .style.format(precision=2)
    .background_gradient(cmap="Blues", subset=["skewness", "kurtosis"])
)

### Comparando "antes" × "depois"

In [ ]:
comparacao = pd.DataFrame({
    "skewness_antes":  desc["skewness"],
    "skewness_depois": desc_depois["skewness"],
    "kurtosis_antes":  desc["kurtosis"],
    "kurtosis_depois": desc_depois["kurtosis"],
})
comparacao["delta_skewness"] = (comparacao["skewness_depois"] - comparacao["skewness_antes"]).round(3)
comparacao["delta_kurtosis"] = (comparacao["kurtosis_depois"] - comparacao["kurtosis_antes"]).round(3)
comparacao.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col in zip(axes, VARS_QUANT_GLOBAL):
    sns.kdeplot(df_antes[col].dropna(), color=COR_CONTEXTO, fill=True, label="Antes", ax=ax)
    sns.kdeplot(df[col].dropna(), color=COR_DESTAQUE, fill=True, label="Depois", ax=ax)
    ax.set_title(col)
    ax.legend()

fig.suptitle("Distribuição Antes × Depois do Tratamento", fontsize=14)
plt.tight_layout()
plt.show()

**Discussão:** As curvas "antes" e "depois" praticamente se sobrepõem? Isso é esperado: removemos menos de ~8% das linhas e não alteramos nenhum outlier. Uma diferença pequena é, neste caso, uma confirmação de que o tratamento foi conservador e adequado — não um sinal de que não adiantou nada.

## 6. Princípio Relacional

### Análise de Variáveis Qualitativas

#### Preparando variáveis qualitativas derivadas

In [ ]:
# Primeiro país e primeiro gênero (colunas originais são multi-valor separadas por vírgula)
df["country_first"] = df["country"].str.split(",").str[0].str.strip()
df["genre_first"]   = df["listed_in"].str.split(",").str[0].str.strip()

VARS_QUALI = ["type", "rating", "country_first", "genre_first"]

df[["title", "country", "country_first", "listed_in", "genre_first"]].head(5)

#### Frequências absolutas e relativas

In [ ]:
for col in VARS_QUALI:
    freq_abs = df[col].value_counts().head(10)
    freq_rel = df[col].value_counts(normalize=True).head(10).round(3)
    print(f"\n--- {col} (top 10) ---")
    print(pd.concat([freq_abs.rename("absoluta"), freq_rel.rename("relativa")], axis=1))

#### Visualizando as Frequências

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
axes = axes.ravel()

top_n = {"type": 2, "rating": 15, "country_first": 12, "genre_first": 12}

for i, col in enumerate(VARS_QUALI):
    dados = df[col].value_counts().head(top_n[col])
    dados.sort_values().plot(kind="barh", color=COR_NEUTRA, ax=axes[i])
    axes[i].set_title(f"Frequência — {col}")
    axes[i].set_xlabel("")

fig.suptitle("Frequências das Variáveis Qualitativas — Netflix", fontsize=14)
plt.tight_layout()
plt.show()

**Discussão:** Quais países dominam o catálogo Netflix? E quais gêneros? Existe concentração em poucos países/gêneros, ou a distribuição é mais equilibrada? Isso pode indicar viés de amostragem (a Netflix tem mais conteúdo de determinadas regiões).

#### Desbalanceamento entre Classes

In [ ]:
def metrica_desbalanceamento(serie):
    contagens = serie.value_counts()
    razao = contagens.max() / contagens.min()
    props = contagens / contagens.sum()
    n_classes = len(contagens)
    ent_norm = entropy(props, base=2) / np.log2(n_classes) if n_classes > 1 else np.nan
    return pd.Series({
        "n_classes": n_classes,
        "razao_desbalanceamento": round(razao, 2),
        "entropia_normalizada": round(ent_norm, 3),
    })

pd.DataFrame({col: metrica_desbalanceamento(df[col]) for col in VARS_QUALI}).T

**Discussão:** `country_first` provavelmente tem entropia baixa — um país domina (EUA?). Isso é um viés estrutural do dataset: a Netflix tem uma produção muito maior de um único país. Como isso afeta análises que cruzam conteúdo por país?

### Análise Bivariada

#### Quantitativa × Quantitativa — Correlação

In [ ]:
corr_pearson  = df[VARS_QUANT_GLOBAL].corr(method="pearson")
corr_spearman = df[VARS_QUANT_GLOBAL].corr(method="spearman")
corr_kendall  = df[VARS_QUANT_GLOBAL].corr(method="kendall")

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, (corr, titulo) in zip(axes, [
    (corr_pearson,  "Pearson"),
    (corr_spearman, "Spearman"),
    (corr_kendall,  "Kendall")
]):
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
    ax.set_title(f"Correlação — {titulo}")

plt.tight_layout()
plt.show()

**Discussão:** Existe correlação entre `release_year` e `year_added`? Seria esperado que títulos mais recentes fossem adicionados mais tarde à plataforma — mas a Netflix também adiciona clássicos antigos. Uma correlação positiva fraca indica isso.

#### Dispersão (scatter) entre `release_year` e `year_added`

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df.dropna(subset=["year_added"]),
                x="release_year", y="year_added",
                hue="type", palette=palette_tipo, alpha=0.4, s=20)
plt.title("Ano de Lançamento × Ano de Adição na Netflix")
plt.xlabel("Ano de Lançamento")
plt.ylabel("Ano de Adição")
plt.tight_layout()
plt.show()

**Discussão:** A nuvem de pontos sugere alguma tendência? Pontos muito abaixo da diagonal (onde `year_added` é muito maior que `release_year`) representam clássicos adicionados recentemente. Filmes e séries ocupam regiões distintas do gráfico?

#### Quantitativa × Qualitativa: Comparando Grupos

Como já vimos que nenhuma variável numérica passou no Shapiro-Wilk, usamos testes **não-paramétricos** como principal (Mann-Whitney U para 2 grupos; Kruskal-Wallis para 3+ grupos), mas calculamos os paramétricos para contraste.

In [ ]:
# release_year: Movie vs TV Show
movies_year = df[df["type"] == "Movie"]["release_year"].dropna()
tv_year     = df[df["type"] == "TV Show"]["release_year"].dropna()

stat_mw, p_mw     = stats.mannwhitneyu(movies_year, tv_year, alternative="two-sided")
stat_tt, p_tt     = stats.ttest_ind(movies_year, tv_year)

print("release_year: Movie vs. TV Show")
print(f"  T-Test (paramétrico)     → estatística={stat_tt:.3f}, p-valor={p_tt:.5g}")
print(f"  Mann-Whitney (não-param) → estatística={stat_mw:.0f}, p-valor={p_mw:.5g}")
print(f"  Médias: Movie={movies_year.mean():.1f} | TV Show={tv_year.mean():.1f}")

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="type", y="release_year", hue="type",
            palette=palette_tipo, legend=False, ax=ax)
ax.set_title(f"release_year por Tipo (Mann-Whitney p={p_mw:.4g})")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
# release_year: por rating (3+ grupos → Kruskal-Wallis)
top_ratings = df["rating"].value_counts().head(8).index.tolist()
df_top_rating = df[df["rating"].isin(top_ratings)]

grupos_rating = [g["release_year"].values for _, g in df_top_rating.groupby("rating")]

stat_anova, p_anova = stats.f_oneway(*grupos_rating)
stat_kw, p_kw       = stats.kruskal(*grupos_rating)

print("release_year entre categorias de 'rating' (top 8)")
print(f"  ANOVA          → estatística={stat_anova:.3f}, p-valor={p_anova:.5g}")
print(f"  Kruskal-Wallis → estatística={stat_kw:.3f}, p-valor={p_kw:.5g}")

plt.figure(figsize=(12, 5))
sns.boxplot(data=df_top_rating, x="rating", y="release_year", color=COR_NEUTRA)
plt.title(f"Ano de Lançamento por Classificação Etária (Kruskal-Wallis p={p_kw:.4g})")
plt.tight_layout()
plt.show()

**Discussão:** O Kruskal-Wallis indica que pelo menos uma categoria de `rating` tem distribuição de `release_year` diferente das demais? Conteúdo infantil (TV-Y, G) tende a ser mais recente ou mais antigo do que conteúdo adulto (TV-MA, R)?

#### Qualitativa × Qualitativa: Qui-Quadrado

In [ ]:
# Associação entre 'rating' e 'type'
contingencia_rating_tipo = pd.crosstab(df["rating"], df["type"])
chi2, p_chi2, gl, esperado = stats.chi2_contingency(contingencia_rating_tipo)

print(f"Qui-Quadrado (rating × type): χ²={chi2:.2f}, gl={gl}, p-valor={p_chi2:.5g}")

# Normalizar por linha para ver proporção por rating
contingencia_norm = contingencia_rating_tipo.div(contingencia_rating_tipo.sum(axis=1), axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(contingencia_rating_tipo, annot=True, fmt="d", cmap="magma", ax=axes[0])
axes[0].set_title(f"rating × type — Contagens\n(Qui² p-valor={p_chi2:.4g})")

sns.heatmap(contingencia_norm.round(2), annot=True, fmt=".2f", cmap="magma", ax=axes[1])
axes[1].set_title("rating × type — Proporções (normalizado por linha)")

plt.tight_layout()
plt.show()

**Discussão:** O p-valor do Qui-Quadrado indica dependência entre `rating` e `type`? Se sim, quais classificações etárias são mais associadas a filmes e quais a séries? Por exemplo, `TV-MA` aparece muito em séries porque a designação já começa com "TV".

#### Análise Temporal: Crescimento do Catálogo ao Longo do Tempo

In [ ]:
conteudo_por_ano = (
    df.dropna(subset=["year_added"])
    .groupby(["year_added", "type"])
    .size()
    .unstack(fill_value=0)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

conteudo_por_ano.plot(kind="bar", color=[COR_CONTEXTO, COR_DESTAQUE], ax=axes[0])
axes[0].set_title("Títulos adicionados por ano (absoluto)")
axes[0].set_xlabel("Ano")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend(title="Tipo")

conteudo_por_ano.div(conteudo_por_ano.sum(axis=1), axis=0).plot(
    kind="bar", stacked=True, color=[COR_CONTEXTO, COR_DESTAQUE], ax=axes[1])
axes[1].set_title("Proporção por tipo ao longo do tempo")
axes[1].set_xlabel("Ano")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend(title="Tipo")

plt.tight_layout()
plt.show()

**Discussão:** Em que anos o catálogo cresceu mais rapidamente? A proporção Movie/TV Show mudou ao longo do tempo — a Netflix está investindo mais em séries nos anos mais recentes?

#### Análise Multivariada — Top Países por Tipo

In [ ]:
top10_paises = df["country_first"].value_counts().head(10).index
df_top10 = df[df["country_first"].isin(top10_paises)]

pais_tipo = (
    df_top10.groupby(["country_first", "type"])
    .size()
    .unstack(fill_value=0)
)

pais_tipo.sort_values("Movie", ascending=True).plot(
    kind="barh", stacked=True, color=[COR_CONTEXTO, COR_DESTAQUE], figsize=(10, 6))
plt.title("Títulos por País de Origem (top 10) — por Tipo")
plt.xlabel("Número de títulos")
plt.legend(title="Tipo")
plt.tight_layout()
plt.show()

In [ ]:
# Top gêneros por tipo
top10_generos = df["genre_first"].value_counts().head(10).index
df_top_generos = df[df["genre_first"].isin(top10_generos)]

genero_tipo = (
    df_top_generos.groupby(["genre_first", "type"])
    .size()
    .unstack(fill_value=0)
)

genero_tipo.sort_values("Movie", ascending=True).plot(
    kind="barh", stacked=True, color=[COR_CONTEXTO, COR_DESTAQUE], figsize=(10, 7))
plt.title("Títulos por Gênero (top 10) — por Tipo")
plt.xlabel("Número de títulos")
plt.legend(title="Tipo")
plt.tight_layout()
plt.show()

**Discussão:** Existem gêneros que são exclusivamente ou predominantemente de um tipo (filmes vs. séries)? Por exemplo, Stand-Up Comedy tende a ser filme; anime/reality show tende a ser série. Isso faz sentido pelo formato do conteúdo?

## Síntese — Principais Conclusões

Com base na EDA realizada no dataset *Netflix Movies and TV Shows*, as **5 principais conclusões** são:

### 1. O catálogo é majoritariamente de filmes, mas séries crescem em proporção
Filmes representam ~69% dos títulos e séries ~31%. No entanto, a análise temporal mostra que a participação de séries (*TV Shows*) aumentou de forma consistente nos anos mais recentes, refletindo a estratégia da Netflix de investir em conteúdo serializado original.

### 2. O catálogo é concentrado em conteúdo recente, com cauda esquerda longa
`release_year` apresenta assimetria negativa: a maioria dos títulos foi lançada após 2000, mas existem clássicos que remontam às décadas de 1920–1980. Essa cauda esquerda é intencional — a Netflix mantém acervos históricos ao lado de lançamentos recentes.

### 3. Os dados ausentes têm mecanismo MAR (Missing At Random) na maioria das colunas
A ausência de `director` (~30%) concentra-se em *TV Shows*, confirmando MAR: a lacuna está relacionada ao tipo de conteúdo, não ao nome do diretor em si. Colunas como `date_added` (~5%) seguem MCAR. A decisão de manter os registros (preenchendo com "Não informado") preservou o volume de dados sem introduzir viés.

### 4. Rating e tipo de conteúdo são estatisticamente dependentes
O teste Qui-Quadrado (p < 0,001) confirma que a classificação etária não é independente do tipo. Classificações com prefixo "TV" (*TV-MA*, *TV-14*, *TV-PG*) dominam nas séries, enquanto *R*, *PG-13* e *PG* predominam nos filmes — uma relação estrutural imposta pelas convenções de classificação de cada mídia.

### 5. Filmes e séries diferem significativamente em ano de lançamento
O teste Mann-Whitney confirma diferença estatisticamente significativa (p < 0,001) entre o ano de lançamento de filmes e séries no catálogo. Séries tendem a ser mais recentes do que os filmes disponíveis, sugerindo que a Netflix prioriza séries novas e mantém filmes mais antigos em maior proporção.

## Declaração de Uso de IA

**Ferramenta utilizada:** Claude (Anthropic) — via interface Claude Code  

**Para o que foi aplicada:**
- Geração da estrutura base do notebook, incluindo a sequência de seções e o código de cada etapa do ciclo EDA (princípios estrutural, distribucional, relacional e tratamento de dados).
- Adaptação do código demonstrado em aula (dataset Heart Disease) para o contexto do dataset Netflix, considerando suas particularidades (variáveis majoritariamente qualitativas, colunas multi-valor, extração de `duration_num` e `year_added`).
- Redação inicial das perguntas de discussão e da seção de síntese.

**O que foi aceito, modificado ou rejeitado e por quê:**
- **Aceito:** a estrutura geral de 6 seções, a convenção de cores (rosa = Movie, roxo = TV Show), os testes estatísticos utilizados (Mann-Whitney, Kruskal-Wallis, Qui-Quadrado) e a separação de `duration_num` por tipo (já que minutos e temporadas são unidades incomparáveis).
- **Modificado:** as perguntas de discussão foram revisadas para que fizessem sentido no contexto real do dataset, verificando se as hipóteses levantadas eram coerentes com o conhecimento de domínio sobre a Netflix.
- **Rejeitado:** uma sugestão inicial de remoção de todos os outliers de `duration_num`, por desconsiderar que curtas-metragens e filmes épicos são gêneros legítimos — a decisão correta foi classificá-los como outliers estatísticos e mantê-los.

## Exportando os Dataframes para o Dashboard

Salva os dataframes original e tratado em data/ para uso no app Streamlit.

In [ ]:
import os

DATA_DIR = os.path.join('..', 'data')
os.makedirs(DATA_DIR, exist_ok=True)

df_antes.to_csv(os.path.join(DATA_DIR, 'netflix_original.csv'), index=False)
df.to_csv(os.path.join(DATA_DIR, 'netflix_tratado.csv'), index=False)

print(f'Exportado: {DATA_DIR}/netflix_original.csv  ({len(df_antes):,} linhas)')
print(f'Exportado: {DATA_DIR}/netflix_tratado.csv   ({len(df):,} linhas)')